In [ ]:
from google.colab import drive
drive.mount('/content/drive')

: 

In [ ]:
!pip install surprise

!pip install numpy==1.26.4

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ***Creating The Data***

In [ ]:
data_customer = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/olist_customers_dataset.csv')
data_order_items = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/olist_order_items_dataset.csv')
data_reviews = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/olist_order_reviews_dataset.csv')
data_order_payments = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/olist_order_payments_dataset.csv')
data_orders = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/olist_orders_dataset.csv')
data_product = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/olist_products_dataset.csv')
data_category = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/product_category_name_translation.csv')

In [ ]:
data_reviews.head()

In [ ]:
merged_data = pd.merge(data_customer , data_orders , on='customer_id', how='left')
merged_data = pd.merge(merged_data , data_order_items , on='order_id', how='left')
merged_data = pd.merge(merged_data , data_product , on='product_id', how='left')
merged_data = pd.merge(merged_data , data_order_payments , on='order_id', how='left')
merged_data = pd.merge(merged_data , data_reviews , on='order_id', how='left')
merged_data = pd.merge(merged_data , data_category , on='product_category_name', how='left')

In [ ]:
merged_data.columns

In [ ]:
lis = [ 'customer_zip_code_prefix',
       'customer_city', 'customer_state','order_status',
       'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'order_item_id',
       'seller_id', 'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'payment_sequential', 'payment_type', 'payment_installments',
       'payment_value' , 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp',
       ]

merged_data.drop(lis , axis=1,inplace=True)

In [ ]:
merged_data.columns

In [ ]:
merged_data.shape

In [ ]:
clean_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/clean_data.csv')
clean_data.columns

In [ ]:
clean_data.shape

In [ ]:
new_data = pd.DataFrame([merged_data['customer_unique_id'] , merged_data['product_id'] , merged_data['review_score'] , clean_data['ReviewCount'] , merged_data['product_category_name_english'] , clean_data['Brand'] , clean_data['Name'] , clean_data['Description'] , clean_data['Tags']])

In [ ]:
new_data.head()

In [ ]:
new_data_transposed = new_data.T
new_data_transposed.to_csv('/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/E-Commerece-Recommendation-System-Machine-Learning-Product-Recommendation-system-datasets/Project_data.csv', index=False)
new_data_transposed.head()

In [ ]:
new_data_transposed.isnull().sum()

In [ ]:
new_data_transposed['review_score'] = new_data_transposed['review_score'].fillna(0)

# ***Data Preprocessing***

In [ ]:
import gdown
import pandas as pd
import re

def load_drive_csv(drive_url, output_filename="dataset.csv"):
    match = re.search(r'/d/([^/]+)', drive_url)
    if not match:
        raise ValueError("Invalid Google Drive URL format.")

    file_id = match.group(1)
    direct_url = f"https://drive.google.com/uc?id={file_id}"

    gdown.download(direct_url, output_filename, quiet=False)

    return pd.read_csv(output_filename)

In [ ]:
drive_link = "https://drive.google.com/file/d/1ZwXoMz9ULYxua8V7d04uD-r6_qIFrAyJ/view?usp=sharing"
df = load_drive_csv(drive_link)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(axis=0)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.loc[df['review_score'] != 0, 'ReviewCount'] = np.random.randint(100, 1001, size=(df['review_score'] != 0).sum())

In [ ]:
df.head()

In [ ]:
df.duplicated().sum()

In [ ]:
column_names = {
    'customer_unique_id' : 'ID' ,
    'review_score' : 'Rating' ,
    'product_category_name_english' : 'Prod_Category' ,

}
df.rename(columns=column_names , inplace=True)


In [ ]:
df['ID'] = df['ID'].str.extract(r'(\d+)').astype(float)
df['product_id'] = df['product_id'].str.extract(r'(\d+)').astype(float)

In [ ]:
df.head()

# ***EDA***

In [ ]:
# Top 50 Popular Products based on
plt.figure(figsize=(20,10))
pop = df['product_id'].value_counts().head(40)
pop.plot(kind='bar' , color='red')
plt.title('Most Popular Products' , size=25)
plt.xlabel('Product ID' , size=18)
plt.ylabel('Number of Occurrences' , size=18)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.show()

In [ ]:
heat_data = df.pivot_table('ID' , 'Rating')

plt.figure(figsize=(8,6))
sns.heatmap(heat_data , annot=True , cmap='coolwarm')
plt.title("Heat Map of User Ratings")
plt.xlabel("Ratings")
plt.xlabel("User ID")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='Rating', data=df, palette='viridis', hue='Rating', legend=True)
plt.title('Distribution of Review Scores')
plt.xlabel('Review Score')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
category_counts = df['Prod_Category'].value_counts().head(20)
category_counts.plot(kind='bar', color='skyblue')
plt.title('Most Frequent Product Categories' , size=25)
plt.xlabel('Product Category' , size=18)
plt.ylabel('Number of Occurrences' , size=18)
plt.xticks(rotation=45, ha='right', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['Rating'], df['ReviewCount'], alpha=0.5)
plt.title('Review Score vs Review Count')
plt.xlabel('Review Score')
plt.ylabel('Review Count')
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(20, 10))
brand_counts = df['Brand'].value_counts().head(20)
brand_counts.plot(kind='bar', color='orange')
plt.title('Most Frequent Brands' , size=25)
plt.xlabel('Brand' , size=18)
plt.ylabel('Number of Occurrences' , size=18)
plt.xticks(rotation=45, ha='right' , fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
average_review_scores = df.groupby('Prod_Category')['Rating'].mean().sort_values(ascending=False)
top_20_categories = average_review_scores.head(20)

plt.figure(figsize=(20, 10))
top_20_categories.plot(kind='bar', color='Green')
plt.title('Average Review Score Based on Product Category' , size=25)
plt.xlabel('Product Category' , size=18)
plt.ylabel('Average Review Score' , size=18)
plt.xticks(rotation=45, ha='right' , fontsize=14)
plt.tight_layout()
plt.show()

# ***Prepare Data for ML Model***


In [ ]:
# Select relevant columns for collaborative filtering
ml_data = df[['ID', 'product_id', 'Rating']].copy()

# Display the first few rows
display(ml_data.tail())

# Get information about the dataframe
ml_data.shape


## Install Libraries and create train , test data

In [ ]:
# Now try importing surprise and loading the data again
import surprise
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

# Define the format with Reader
reader = Reader(rating_scale=(1, 5))

# Load the data from the pandas DataFrame
data = Dataset.load_from_df(ml_data[['ID', 'product_id', 'Rating']], reader)

# Split data into training and testing sets
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
print(f"Number of users in the trainset: {trainset.n_users}")
print(f"Number of items in the trainset: {trainset.n_items}")

# ***Choose Multiple ML Models***

In [ ]:
from surprise import SVD, NMF, BaselineOnly, KNNBasic , accuracy


In [ ]:
from surprise import SVD, NMF, BaselineOnly, KNNBasic, CoClustering, NormalPredictor, KNNWithMeans, KNNWithZScore, accuracy

models = {
    'SVD': SVD(),
    'NMF': NMF(),
    'BaselineOnly': BaselineOnly(),
    'KNNBasic': KNNBasic(),
    'CoClustering': CoClustering(),
}

results = {}

for name, model in models.items():
    print(f"Training and evaluating {name}...")
    # Train the model
    model.fit(trainset)
    # Make predictions on the test set
    predictions = model.test(testset)
    # Calculate RMSE and MAE
    rmse = accuracy.rmse(predictions, verbose=False)
    mae = accuracy.mae(predictions, verbose=False)
    results[name] = {'RMSE': rmse, 'MAE': mae}
    print(f"{name} - RMSE: {rmse:.4f}, MAE: {mae:.4f}")

## Compare Model Performance

In [ ]:
# Display the performance results
print("Model Performance Comparison:")
for name, metrics in results.items():
    print(f"{name}:")
    print(f"  RMSE: {metrics['RMSE']:.4f}")
    print(f"  MAE: {metrics['MAE']:.4f}")

# You can also visualize the results
import matplotlib.pyplot as plt

model_names = list(results.keys())
rmse_values = [metrics['RMSE'] for metrics in results.values()]
mae_values = [metrics['MAE'] for metrics in results.values()]

x = range(len(model_names))

plt.figure(figsize=(10, 6))
plt.bar(x, rmse_values, width=0.4, label='RMSE', color='skyblue')
plt.bar([i + 0.4 for i in x], mae_values, width=0.4, label='MAE', color='lightcoral')
plt.xticks([i + 0.2 for i in x], model_names, rotation=45, ha='right')
plt.ylabel('Error')
plt.title('Comparison of Model Performance (RMSE and MAE)')
plt.legend()
plt.show()

In [ ]:
from surprise.model_selection import cross_validate

best_model = SVD()

cv_results = cross_validate(best_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

print("\nCross-validation results:")
print(f"Average RMSE: {cv_results['test_rmse'].mean():.4f}")
print(f"Average MAE: {cv_results['test_mae'].mean():.4f}")

print("\nResults for each fold:")
print(cv_results)

In [ ]:
from surprise.model_selection import GridSearchCV
from surprise import SVD

# Define the parameter grid to tune
# You can add or change parameters and their values based on your needs
param_grid = {
    'n_epochs': [20, 30],  # Number of iterations
    'lr_all': [0.002, 0.005], # Learning rate for all parameters
    'reg_all': [0.02, 0.1] # Regularization term for all parameters
}

# Use GridSearchCV to find the best parameters
# We'll evaluate using RMSE and MAE
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3) # Using 3-fold cross-validation for speed

gs.fit(data)

# Print the best RMSE score
print("Best RMSE score:", gs.best_score['rmse'])

# Print the corresponding parameters for the best RMSE score
print("Best parameters for RMSE:", gs.best_params['rmse'])

# Print the best MAE score
print("Best MAE score:", gs.best_score['mae'])

# Print the corresponding parameters for the best MAE score
print("Best parameters for MAE:", gs.best_params['mae'])

In [ ]:
from surprise import SVD

# Define the best hyperparameters based on the Grid Search results for RMSE
best_params = gs.best_params['rmse']

# Initialize the SVD model with the best hyperparameters
best_svd_model = SVD(n_epochs=best_params['n_epochs'],
                     lr_all=best_params['lr_all'],
                     reg_all=best_params['reg_all'])

# Train the model on the entire dataset (or trainset if you prefer)
# Training on the entire data is common after finding best parameters
print("Training SVD model with best hyperparameters...")
best_svd_model.fit(data.build_full_trainset())

print("Training complete!")

# ***Generate Recommendations***

In [ ]:
from collections import defaultdict

def get_top_n_recommendations(predictions, n=10):
    # First map the predictions to each user.
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))

    # Then sort the predictions for each user and retrieve the k top ones.
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]

    return top_n

# Choose the best performing model (based on RMSE in this case)
best_model_name = min(results, key=lambda k: results[k]['RMSE'])
best_model = models[best_model_name]

print(f"Using the best performing model: {best_model_name}")

# Get recommendations for a sample user using the best model
# Select a random sample user ID from the data
sample_user_id = np.random.choice(ml_data['ID'].unique())

# Get all product IDs that the sample user has not rated yet
all_product_ids = ml_data['product_id'].unique()
rated_product_ids = ml_data[ml_data['ID'] == sample_user_id]['product_id'].unique()
unrated_product_ids = [product_id for product_id in all_product_ids if product_id not in rated_product_ids]

# Predict ratings for the unrated products for the sample user
predictions_for_sample_user = [best_model.predict(sample_user_id, product_id) for product_id in unrated_product_ids]

# Get the top N recommendations for the sample user
top_n_recommendations_best_model = get_top_n_recommendations(predictions_for_sample_user, n=10)

# Print the recommended items
print(f"\nTop 10 recommendations for user {sample_user_id} using {best_model_name}:")
if sample_user_id in top_n_recommendations_best_model:
    for product_id, estimated_rating in top_n_recommendations_best_model[sample_user_id]:
        print(f"  Product ID: {product_id}, Estimated Rating: {estimated_rating:.2f}")
else:
    print(f"  No recommendations found for user {sample_user_id}.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from surprise import Prediction # Import Prediction

# Keep the TfidfVectorizer and tfidf_matrix_content creation outside the function
# if the dataframe doesn't change frequently, to avoid recomputing every time.
# However, for demonstration purposes within a function call, we'll keep it here.
# We will initialize the vectorizer outside the function if the dataframe is loaded.
tfidf_vectorizer = None
tfidf_matrix_content = None

def initialize_vectorizer(df):
    """Initializes the TF-IDF vectorizer and matrix."""
    global tfidf_vectorizer, tfidf_matrix_content
    if df is not None and not df.empty:
        # Combine relevant text columns for TF-IDF vectorization
        # Giving more weight to 'Name' and 'Brand' by repeating them
        df['combined_text'] = df['Prod_Category'].fillna('') + (' ' + df['Brand'].fillna(''))*3 + (' ' + df['Name'].fillna(''))*3 + ' ' + \
                              df['Description'].fillna('') + ' ' + \
                              df['Tags'].fillna('')

        tfidf_vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix_content = tfidf_vectorizer.fit_transform(df['combined_text'])
        print("TF-IDF vectorizer and matrix initialized using combined text with weighted fields.")
        print(f"Shape of TF-IDF matrix: {tfidf_matrix_content.shape}")
    else:
        print("Cannot initialize TF-IDF vectorizer: DataFrame is empty.")


def Content_Base_Recomendation(df , search_term , top_n=10):
  print(f"Search term: '{search_term}'")
  if df is None or df.empty:
      print("Error: DataFrame is empty.")
      return pd.DataFrame()

  global tfidf_vectorizer, tfidf_matrix_content
  # Ensure vectorizer and matrix are initialized
  if tfidf_vectorizer is None or tfidf_matrix_content is None:
      initialize_vectorizer(df)
      # If still not initialized, return empty DataFrame
      if tfidf_vectorizer is None or tfidf_matrix_content is None:
          return pd.DataFrame()


  # Vectorize the search term using the *fitted* vectorizer
  search_vector = tfidf_vectorizer.transform([search_term])
  print(f"Shape of search vector: {search_vector.shape}")


  # Calculate cosine similarity between the search term vector and all product tag vectors
  cos_sim = cosine_similarity(search_vector, tfidf_matrix_content)
  print(f"Shape of cosine similarity matrix: {cos_sim.shape}")

  # Get the similarity scores for the search term
  similar_items = list(enumerate(cos_sim[0]))
  print(f"Number of similar items initially: {len(similar_items)}")


  # Sort the predictions and retrieve the k top ones.
  similar_items = sorted(similar_items , key=lambda x:x[1], reverse=True)
  print(f"Top {top_n} similar items (index, score): {similar_items[:top_n]}")


  # Exclude items with 0.0 similarity and take the top_n
  Top_similar_items = [item for item in similar_items if item[1] > 0.0][:top_n]


  recomended_indexes = [x[0] for x in Top_similar_items]
  print(f"Recommended indexes: {recomended_indexes}")

  if not recomended_indexes:
      print("No items found with similarity score greater than 0.0.")
      return pd.DataFrame()

  # Ensure recommended_indexes are within the valid range of the dataframe index
  valid_recomended_indexes = [idx for idx in recomended_indexes if idx < len(df)]
  print(f"Valid recommended indexes: {valid_recomended_indexes}")

  if not valid_recomended_indexes:
      print("No valid recommended indexes found within DataFrame bounds after filtering.")
      return pd.DataFrame()


  recomended_items = df.iloc[valid_recomended_indexes][['Name' , 'Brand' , 'ReviewCount' , 'Rating']]
  print(f"Shape of recommended items DataFrame: {recomended_items.shape}")


  return recomended_items

# You may need to call initialize_vectorizer(df) after loading your dataframe
# For example, after cell PmU1dXg-LcP8

## ***Search based on list of Items***

In [ ]:
def initialize_vectorizer(df, ngram_range=(1, 1), min_df=0.001, max_df=0.999):
    """Initializes the TF-IDF vectorizer and matrix."""
    global tfidf_vectorizer, tfidf_matrix_content
    if df is not None and not df.empty:
        # Combine relevant text columns for TF-IDF vectorization
        df['combined_text'] = df['Prod_Category'].fillna('') + (' ' + df['Brand'].fillna(''))*3 + (' ' + df['Name'].fillna(''))*3 + ' ' + \
                              df['Description'].fillna('') + ' ' + \
                              df['Tags'].fillna('')

        tfidf_vectorizer = TfidfVectorizer(stop_words='english', ngram_range=ngram_range, min_df=min_df, max_df=max_df)
        tfidf_matrix_content = tfidf_vectorizer.fit_transform(df['combined_text'])
        print(f"TF-IDF vectorizer and matrix initialized using combined text with weighted fields and ngram_range={ngram_range}.")
        print(f"Shape of TF-IDF matrix: {tfidf_matrix_content.shape}")
    else:
        print("Cannot initialize TF-IDF vectorizer: DataFrame is empty.")

# Re-initialize with n-grams (e.g., unigrams and bigrams) and previously found good min_df/max_df

In [ ]:
from surprise import Prediction
from collections import defaultdict

def hybrid_recommendations(user_id, search_term=None, cf_model=None, cb_function=None, dataframe=None, history_threshold=5, top_n=10):
    # Ensure TF-IDF vectorizer and matrix are initialized before potentially using CB
    global tfidf_vectorizer, tfidf_matrix_content
    if tfidf_vectorizer is None or tfidf_matrix_content is None:
        print("TF-IDF vectorizer and matrix not initialized. Initializing now.")
        initialize_vectorizer(dataframe) # Pass the dataframe to the initialization

    # Check if the user is in the CF training set
    user_rated_items = dataframe[dataframe['ID'] == user_id]

    if len(user_rated_items) >= history_threshold and cf_model:
        print(f"User {user_id} is a CF user.")
        # Collaborative Filtering approach
        if search_term is not None and cb_function is not None:
            print(f"Using hybrid approach for CF user with search term '{search_term}'.")
            # Hybrid: CF + CB based on a search term
            cb_recs = cb_function(dataframe, search_term, top_n=top_n * 2) # Pass the dataframe to CB function
            if not cb_recs.empty:
                # Get CF predictions for the content-based recommended items
                cb_item_ids = []
                for index, row in cb_recs.iterrows():
                    # Find the product_id in the original dataframe based on Name and Brand
                    matching_items = dataframe[(dataframe['Name'] == row['Name']) & (dataframe['Brand'] == row['Brand'])]
                    if not matching_items.empty:
                        cb_item_ids.append(matching_items.iloc[0]['product_id'])

                cf_predictions = [cf_model.predict(user_id, iid) for iid in cb_item_ids if not pd.isna(iid)] # Filter out NaN product_ids
                # Combine scores (simple weighted average or just use CF score if available)
                # For simplicity, we'll just use the estimated CF rating if available
                recommended_items = []
                for pred in cf_predictions:
                     # Check if the item_id from prediction exists in the dataframe
                    if pred.iid in dataframe['product_id'].values:
                        item_details = dataframe[dataframe['product_id'] == pred.iid].iloc[0]
                        recommended_items.append({
                            'Name': item_details['Name'],
                            'Brand': item_details['Brand'],
                            'ReviewCount': item_details['ReviewCount'],
                            'Rating': pred.est # Use the estimated rating from CF
                        })
                recommended_df = pd.DataFrame(recommended_items)
                # Sort by estimated rating and return top N
                return recommended_df.sort_values(by='Rating', ascending=False).head(top_n)
            else:
                print("No content-based recommendations found for the search term.")
                return pd.DataFrame()
        else:
            print(f"Using CF approach for user {user_id} (no search term specified).")
            # Pure Collaborative Filtering
            # Get a list of all product IDs
            all_product_ids = dataframe['product_id'].unique()
            # Get a list of product IDs rated by the user
            rated_product_ids = user_rated_items['product_id'].unique()
            # Get a list of product IDs not rated by the user
            unrated_product_ids = [pid for pid in all_product_ids if pid not in rated_product_ids and not pd.isna(pid)] # Filter out NaN product_ids

            # Predict ratings for unrated items
            predictions = [cf_model.predict(user_id, iid) for iid in unrated_product_ids]

            # Get the top N recommendations based on predicted ratings
            top_predictions = sorted(predictions, key=lambda x: x.est, reverse=True)[:top_n]

            recommended_items = []
            for pred in top_predictions:
                 # Check if the item_id from prediction exists in the dataframe
                if pred.iid in dataframe['product_id'].values:
                    item_details = dataframe[dataframe['product_id'] == pred.iid].iloc[0]
                    recommended_items.append({
                        'Name': item_details['Name'],
                        'Brand': item_details['Brand'],
                        'ReviewCount': item_details['ReviewCount'],
                        'Rating': pred.est # Use the estimated rating from CF
                    })
            return pd.DataFrame(recommended_items)

    elif search_term is not None and cb_function is not None:
        print(f"User {user_id} is a cold-start user or has few ratings. Using content-based approach with search term '{search_term}'.")
        # Cold Start user: Content-Based approach based on a search term
        return cb_function(dataframe, search_term, top_n=top_n) # Pass the dataframe to CB function

    else:
        print(f"Cannot generate recommendations for user {user_id}. User has few ratings and no search term was provided.")
        return pd.DataFrame()

In [ ]:

# Define different min_df and max_df value combinations to experiment with
param_combinations = [
    (0.01, 0.99),
    (0.05, 0.95),
    (0.001, 0.999), #Giving Best Outputs
    (0.005, 0.995),
    (0.01, 0.95),
    (0.05, 0.99)
]

# Create a loop that iterates through the defined combinations
for min_df_val, max_df_val in param_combinations:
    print(f"\n--- Trying min_df={min_df_val}, max_df={max_df_val} ---")

    # Re-initialize the TfidfVectorizer with the current min_df and max_df values
    # and fit it to the combined_text column
    tfidf_vectorizer = TfidfVectorizer(stop_words='english', min_df=min_df_val, max_df=max_df_val)
    df['combined_text'] = df['Prod_Category'].fillna('') + (' ' + df['Brand'].fillna(''))*3 + (' ' + df['Name'].fillna(''))*3 + ' ' + \
                          df['Description'].fillna('') + ' ' + \
                          df['Tags'].fillna('')
    tfidf_matrix_content = tfidf_vectorizer.fit_transform(df['combined_text'])
    print(f"TF-IDF matrix shape for this combination: {tfidf_matrix_content.shape}")

    # Define a few sample search terms for evaluation
    sample_search_terms = [
        "Xiaomi",
        "laptop",
        "makeup",
        "sports shoe",
        "furniture",
        "electronics",
        "baby products",
    ]

    # For each combination, generate content-based recommendations for sample search terms
    for term in sample_search_terms:
        print(f"\nRecommendations for search term: '{term}'")
        recommendations = Content_Base_Recomendation(df, term, top_n=5) # Get top 5 for brevity
        if not recommendations.empty:
            display(recommendations)
        else:
            print("No recommendations found for this search term and parameter combination.")

In [ ]:
df.head()

# ***Give Input Get Recommendations***

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english', min_df=0.001, max_df=0.999)
tfidf_matrix_content = tfidf_vectorizer.fit_transform(df['combined_text'])

In [ ]:
input = 'Clairol Nice N Easy Permanent' # Give the input

print(f"TF-IDF matrix shape for this combination: {tfidf_matrix_content.shape}")
recommendations = Content_Base_Recomendation(df, input, top_n=10) # Get top 5 for brevity
display(recommendations)

In [ ]:

sample_user_id_for_recommendations = 861.0

search_term_for_recommendations = "makeup"

recommendations = hybrid_recommendations(
    user_id=sample_user_id_for_recommendations,
    search_term=search_term_for_recommendations,
    cf_model=best_svd_model,
    cb_function=Content_Base_Recomendation,
    dataframe=df,
    top_n=10 # Number of recommendations to generate
)

if not recommendations.empty:
    print(f"\nRecommendations for user {sample_user_id_for_recommendations} (and search term '{search_term_for_recommendations}'):")
    display(recommendations)
else:
    print(f"\nCould not generate recommendations for user {sample_user_id_for_recommendations} (and search term '{search_term_for_recommendations}').")

# ***Download Best model***

In [ ]:
from surprise.dump import dump

# Define filename for saving the model
filename = '/content/drive/MyDrive/Colab Notebooks/Datasets /Project_dataset/best_model.pkl'

# Save the model
dump(filename, algo=best_model)

print(f"Best model '{best_model_name}' saved to {filename}")

In [ ]:
from google.colab import files
files.download('recommendation_model.pkl')
